# 1 · An operator, taken apart

Companion to the tutorial *From Hand-Crafted to LLM-Based Variation Operators in Metaheuristics*. It runs offline: the model is a fixed pool of completions, so every number below comes out the same on your machine.

A metaheuristic varies a candidate and decides whether to keep it. Replace the
variation step with a call to a language model and four things have to happen
that did not have to happen before. This notebook does each of them by hand,
one cell at a time, and only then shows the loop that does all four.

By the end you will have run a complete operator on a problem small enough to
check in your head, and you will know why the loop keeps two solutions instead
of one.

In [1]:
import _bootstrap  # puts the repository root on sys.path
import viz

viz.anatomy()

'<svg xmlns="http://www.w3.org/2000/svg" width="660" height="216" viewBox="0 0 660 216" font-family="ui-sans-serif,-apple-system,Segoe UI,Roboto,sans-serif"><rect width="660" height="216" fill="#ffffff"/><defs><marker id="a" markerWidth="7" markerHeight="7" refX="6" refY="3.5" orient="auto"><path d="M0,0 L7,3.5 L0,7 z" fill="#8a8f9a"/></marker></defs><text x="20.0" y="20.0" font-size="13" fill="#1f2430" text-anchor="start" font-weight="600">One step of an LLM variation operator</text><rect x="20" y="74" width="138" height="62" rx="8" fill="#2f6fdb" fill-opacity="0.07" stroke="#2f6fdb" stroke-width="1.4"/><text x="89.0" y="98.0" font-size="12" fill="#1f2430" text-anchor="middle" font-weight="600">prompt</text><text x="89.0" y="114.0" font-size="10" fill="#8a8f9a" text-anchor="middle" font-weight="normal">what conditions</text><text x="89.0" y="127.0" font-size="10" fill="#8a8f9a" text-anchor="middle" font-weight="normal">the next sample</text><line x1="161" y1="105.0" x2="175" y2="105.0" stroke="#8a8f9a" stroke-width="1.4" marker-end="url(#a)"/><rect x="180" y="74" width="138" height="62" rx="8" fill="#1f2430" fill-opacity="0.07" stroke="#1f2430" stroke-width="1.4"/><text x="249.0" y="98.0" font-size="12" fill="#1f2430" text-anchor="middle" font-weight="600">sample</text><text x="249.0" y="114.0" font-size="10" fill="#8a8f9a" text-anchor="middle" font-weight="normal">one model call,</text><text x="249.0" y="127.0" font-size="10" fill="#8a8f9a" text-anchor="middle" font-weight="normal">returns text</text><line x1="321" y1="105.0" x2="335" y2="105.0" stroke="#8a8f9a" stroke-width="1.4" marker-end="url(#a)"/><rect x="340" y="74" width="138" height="62" rx="8" fill="#d98324" fill-opacity="0.07" stroke="#d98324" stroke-width="1.4"/><text x="409.0" y="98.0" font-size="12" fill="#1f2430" text-anchor="middle" font-weight="600">parse + validate</text><text x="409.0" y="114.0" font-size="10" fill="#8a8f9a" text-anchor="middle" font-weight="normal">schema, syntax,</text><text x="409.0" y="127.0" font-size="10" fill="#8a8f9a" text-anchor="middle" font-weight="normal">feasibility</text><line x1="481" y1="105.0" x2="495" y2="105.0" stroke="#8a8f9a" stroke-width="1.4" marker-end="url(#a)"/><rect x="500" y="74" width="138" height="62" rx="8" fill="#1f9d78" fill-opacity="0.07" stroke="#1f9d78" stroke-width="1.4"/><text x="569.0" y="98.0" font-size="12" fill="#1f2430" text-anchor="middle" font-weight="600">evaluate + accept</text><text x="569.0" y="114.0" font-size="10" fill="#8a8f9a" text-anchor="middle" font-weight="normal">score it, keep it</text><text x="569.0" y="127.0" font-size="10" fill="#8a8f9a" text-anchor="middle" font-weight="normal">or walk away</text><path d="M 409.0 74 C 409.0 40, 89.0 40, 89.0 72" fill="none" stroke="#d98324" stroke-width="1.4" stroke-dasharray="4 3" marker-end="url(#a)"/><text x="249.0" y="36.0" font-size="10" fill="#d98324" text-anchor="middle" font-weight="normal">invalid: repair, bounded</text><path d="M 569.0 136 C 569.0 176, 89.0 176, 89.0 138" fill="none" stroke="#1f9d78" stroke-width="1.4" stroke-dasharray="4 3" marker-end="url(#a)"/><text x="329.0" y="190.0" font-size="10" fill="#1f9d78" text-anchor="middle" font-weight="normal">next step, carrying the feedback</text></svg>'

## The vocabulary, once

**Artifact.** Whatever the operator produces: a solution, a heuristic, a program.

**Representation `r`.** The text placed in the prompt at generation time. This is
the design variable of the whole tutorial. Change `r` and you change what the
operator proposes.

**Proposal distribution `q(x' | r)`.** The operator seen from outside: a machine
that turns a representation into a distribution over candidates. A classical
mutation is also one of these, with a fixed kernel you wrote by hand.

Nothing here requires the model to understand anything. It samples token
sequences; the rest of the machinery turns those into artifacts or refuses them.

## Step one: what comes back is text

The first surprise when you build one of these is that the model does not return
a solution. It returns a string, and the string has to survive a trip through a
parser before the search can do anything with it.

Here is a real completion, the first one the mock model returns in the TSP demo:

In [2]:
from llm import MockLLM, load_pool

pool = load_pool('tsp_pool.txt')
print(pool[0])

CANDIDATE
id: t
representation: permutation
payload:
[0, 1, 4, 4, 2]
END_CANDIDATE


`CANDIDATE ... END_CANDIDATE` is the contract. It exists because a model that is
asked for a tour will often also say hello, explain itself, or wrap the answer in
a fenced code block. An envelope gives the parser something to look for, and it
gives the validator its first cheap check: no envelope, no candidate, and nothing
else runs.

`representation:` says which schema the payload follows. `payload:` carries it.
That is the whole convention, and it is the same one the bin-packing operator
uses for code later on.

## Step two: the four functions a problem has to supply

The loop is written once and reused. A new problem plugs in through an object
with four methods, and nothing else:

| method | question it answers |
|---|---|
| `render(cur, score, feedback)` | what goes in the prompt this step |
| `parse(text)` | is there an artifact in this reply, and what is it |
| `feasible(artifact)` | is it legal in this problem |
| `repair_prompt(prompt, text, err)` | what to send back when it is not |

Below is one for a problem with no domain at all: the artifact is an integer and
smaller is better. It is written here because it exists nowhere else in the
repository. Everything else in these notebooks is imported.

In [3]:
class IntSpec:
    """Toy problem: propose a smaller non-negative integer."""

    def render(self, cur, score, feedback):
        # feedback entries are (kind, info) pairs; the log the loop returns adds
        # the step number in front, so the two shapes are not the same tuple
        history = ' | '.join(f'{k}:{v}' for k, v in feedback[-3:]) or 'nothing yet'
        return (f'[context] find a small non-negative integer\n'
                f'[conditioning] incumbent {cur}, score {score}\n'
                f'[history] {history}\n'
                f'[instruction] emit one smaller integer\n'
                f'[format] CANDIDATE envelope, payload is the integer')

    @staticmethod
    def parse(text):
        if 'CANDIDATE' not in text:
            raise ValueError('schema error: no CANDIDATE envelope')
        body = text.split('payload:')[1].split('END_CANDIDATE')[0].strip()
        try:
            return int(body)
        except ValueError:
            raise ValueError(f'syntax error: {body!r} is not an integer')

    @staticmethod
    def feasible(v):
        if v < 0:
            raise ValueError(f'feasibility: {v} is negative')
        return True

    def repair_prompt(self, prompt, text, err):
        return prompt + f'\n[repair] the previous reply failed validation: {err}'

spec = IntSpec()
print('four methods:', [m for m in dir(spec) if not m.startswith('_')])

four methods: ['feasible', 'parse', 'render', 'repair_prompt']


## Step three: one step, by hand

Before letting the loop run, here is exactly what it does in a single step. Four
cells, four events.

In [4]:
from llm import envelope

cur = 20
evaluate = lambda v: float(v)

prompt = spec.render(cur, evaluate(cur), [])
print(prompt)

[context] find a small non-negative integer
[conditioning] incumbent 20, score 20.0
[history] nothing yet
[instruction] emit one smaller integer
[format] CANDIDATE envelope, payload is the integer


In [5]:
llm = MockLLM([envelope('12', ident='n', representation='integer'),
               envelope('-3', ident='n', representation='integer'),
               envelope('4',  ident='n', representation='integer')])

reply = llm.sample(prompt)
print(reply)

CANDIDATE
id: n
representation: integer
payload:
12
END_CANDIDATE


In [6]:
candidate = spec.parse(reply)      # layers 1 and 2: envelope, then syntax
spec.feasible(candidate)           # layer 3: legal in this problem
score = evaluate(candidate)
print(f'candidate {candidate}, score {score}')

candidate 12, score 12.0


In [7]:
accept = lambda cand, sc, cur, scur: sc < scur   # strict improvement

if accept(candidate, score, cur, evaluate(cur)):
    cur = candidate
    print(f'accepted, the incumbent is now {cur}')
else:
    print(f'rejected, the incumbent stays at {cur}')

accepted, the incumbent is now 12


That is the operator. Prompt, sample, validate, decide. Everything else in this
repository is those four events with a harder problem plugged in.

## Step four: the loop that repeats it

`search.py` holds exactly one copy of Algorithm 1, and this is it, printed from
the module rather than pasted here. Read the inner `for` first: that is bounded
repair, and notebook 2 is about it.

In [8]:
import inspect
from search import build_and_validate

print(inspect.getsource(build_and_validate))

def build_and_validate(llm, evaluator: Callable[[Any], float],
                       accept: Callable[[Any, float, Any, float], bool],
                       incumbent: Any, spec, budget: int, retries: int,
                       minimize: bool = True) -> Tuple[Any, float, List[tuple]]:
    cur = best = incumbent
    best_score = evaluator(incumbent)
    feedback: List[tuple] = []
    log: List[tuple] = []
    for t in range(budget):
        prompt = spec.render(cur, evaluator(cur), feedback)
        ok, cand, err = False, None, None
        for _ in range(retries + 1):
            text = llm.sample(prompt)
            try:
                cand = spec.parse(text)        # schema + syntax
                spec.feasible(cand)            # domain feasibility
                ok = True
                break
            except ValueError as exc:
                err = str(exc)
                prompt = spec.repair_prompt(prompt, text, err)
        if not ok:
            feedback.append(("inval

In [9]:
llm = MockLLM([envelope('12', ident='n', representation='integer'),
               envelope('-3', ident='n', representation='integer'),  # infeasible
               envelope('4',  ident='n', representation='integer')])

best, best_score, log = build_and_validate(
    llm, evaluator=evaluate, accept=accept,
    incumbent=20, spec=spec, budget=3, retries=0, minimize=True)

for step, kind, info in log:
    print(f'step {step}: {kind:8s} {info}')
print(f'best {best}, score {best_score}')

step 0: accepted 12.0
step 1: invalid  feasibility: -3 is negative
step 2: accepted 4.0
best 4, score 4.0


In [10]:
viz.trajectory(20.0, log, 'Three steps on the toy problem')

'<svg xmlns="http://www.w3.org/2000/svg" width="620" height="214" viewBox="0 0 620 214" font-family="ui-sans-serif,-apple-system,Segoe UI,Roboto,sans-serif"><rect width="620" height="214" fill="#ffffff"/><text x="20.0" y="18.0" font-size="13" fill="#1f2430" text-anchor="start" font-weight="600">Three steps on the toy problem</text><line x1="46" y1="172.0" x2="602" y2="172.0" stroke="#d8dce3" stroke-width="1"/><text x="36.0" y="176.0" font-size="10" fill="#8a8f9a" text-anchor="end" font-weight="normal">4</text><line x1="46" y1="106.0" x2="602" y2="106.0" stroke="#d8dce3" stroke-width="1"/><text x="36.0" y="110.0" font-size="10" fill="#8a8f9a" text-anchor="end" font-weight="normal">12</text><line x1="46" y1="40.0" x2="602" y2="40.0" stroke="#d8dce3" stroke-width="1"/><text x="36.0" y="44.0" font-size="10" fill="#8a8f9a" text-anchor="end" font-weight="normal">20</text><polyline points="46.0,40.0 231.3,106.0 416.7,106.0 602.0,172.0" fill="none" stroke="#1f2430" stroke-width="1.6" stroke-opacity="0.35"/><circle cx="46.0" cy="40.0" r="5" fill="#ffffff" stroke="#1f2430" stroke-width="1.6"/><text x="46.0" y="190.0" font-size="10" fill="#8a8f9a" text-anchor="middle" font-weight="normal">start</text><circle cx="231.3" cy="106.0" r="5.5" fill="#1f9d78" fill-opacity="0.9"/><text x="231.3" y="190.0" font-size="10" fill="#8a8f9a" text-anchor="middle" font-weight="normal">step 0</text><circle cx="416.7" cy="172.0" r="5.5" fill="#d98324" fill-opacity="0.9"/><text x="416.7" y="190.0" font-size="10" fill="#8a8f9a" text-anchor="middle" font-weight="normal">step 1</text><circle cx="602.0" cy="172.0" r="5.5" fill="#1f9d78" fill-opacity="0.9"/><text x="602.0" y="190.0" font-size="10" fill="#8a8f9a" text-anchor="middle" font-weight="normal">step 2</text><circle cx="46" cy="204" r="4.5" fill="#1f9d78"/><text x="55.0" y="208.0" font-size="10" fill="#8a8f9a" text-anchor="start" font-weight="normal">accepted</text><circle cx="142" cy="204" r="4.5" fill="#d9456b"/><text x="151.0" y="208.0" font-size="10" fill="#8a8f9a" text-anchor="start" font-weight="normal">rejected</text><circle cx="238" cy="204" r="4.5" fill="#d98324"/><text x="247.0" y="208.0" font-size="10" fill="#8a8f9a" text-anchor="start" font-weight="normal">invalid</text><text x="602.0" y="18.0" font-size="10" fill="#8a8f9a" text-anchor="end" font-weight="normal">lower is better</text></svg>'

Three steps, three different outcomes, and the difference between them matters.

- **accepted**: the candidate was valid and the rule took it.
- **invalid**: the validator refused it, so it was never scored. The retry budget
  was zero here, so the step produced nothing.
- **rejected**: valid, scored, and the rule turned it down.

A log that only said *failed* would hide the distinction, and the distinction is
what tells you whether to fix the prompt, the validator or the acceptance rule.

## Why the loop keeps two solutions

`cur` is where the search is standing. `best` is the best artifact it has ever
seen. With a strict-improvement rule they are the same thing, which is why the
separation looks redundant until you change the rule.

Accept everything, the way a simulated-annealing style rule sometimes does, and
the walk goes downhill on purpose:

In [11]:
llm = MockLLM([envelope('3', ident='n', representation='integer'),
               envelope('40', ident='n', representation='integer')])

best, best_score, log = build_and_validate(
    llm, evaluator=evaluate, accept=lambda *a: True,   # take every candidate
    incumbent=20, spec=spec, budget=2, retries=0, minimize=True)

print('log :', [(k, v) for _, k, v in log])
print('the walk ended at 40, and best is', best)

log : [('accepted', 3.0), ('accepted', 40.0)]
the walk ended at 40, and best is 3


The walk ends on 40. `best` is 3. One line in the loop, and the operator stays
correct under a rule that accepts worsening moves.

## Try it

Change one thing at a time and re-run the cell above.

1. Set `retries=1` and put an unparseable reply first in the pool. The step that
   was *invalid* becomes *accepted*.
2. Make `accept` take equal scores as well. Watch the log fill with accepted steps
   that do not move `best`.
3. Add a fourth `[history]` line to `render` and print the prompt at step 3. That
   is the feedback channel, and it is the subject of notebook 5.

---

Next: [2 · Refusing a reply, and saying why](02_validate_and_repair.ipynb)